In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer, RobustScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.cluster import KMeans
import warnings
import re
import os

In [2]:
plt.style.use('ggplot')
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')
plt.rcParams.update({
    'font.sans-serif': ['Source Han Sans CN'],  # 关键：用排查到的思源黑体
    'axes.unicode_minus': False,                # 解决负号显示为方块的问题
    'figure.dpi': 100,                          # 高清渲染
    'savefig.dpi': 300,                         # 保存图片高清
    'savefig.facecolor': 'white'                # 云平台保存图片背景白色
})


class Config:
    seed = 2026 # 更换种子以增加差异性
    n_folds = 5 # 5折交叉验证
    
    # 硬件配置
    LGBM_DEVICE = 'cpu'
    XGB_DEVICE = 'gpu_hist' if torch.cuda.is_available() else 'hist'
    NN_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # NN 超参数
    NN_EPOCHS = 60
    NN_BATCH_SIZE = 256
    NN_LR = 8e-4

    # 列名映射
    col_mapping_rent = {
        '户型': '房屋户型',
        '楼层': '所在楼层',
        '面积': '建筑面积',
        '朝向': '房屋朝向',
        '结构': '建筑结构',
        '环线位置': '环线' 
    }

In [3]:

def load_data():
    train_price = pd.read_csv('/home/mw/input/hackathon255769/ruc_Class25Q2_train_price.csv')
    train_rent = pd.read_csv('/home/mw/input/hackathon255769/ruc_Class25Q2_train_rent.csv')
    test_price = pd.read_csv('/home/mw/input/hackathon255769/ruc_Class25Q2_test_price.csv')
    test_rent = pd.read_csv('/home/mw/input/hackathon255769/ruc_Class25Q2_test_rent.csv')
    
    # 标记数据集
    train_price['dataset'] = 'train_price'
    test_price['dataset'] = 'test_price'
    train_rent['dataset'] = 'train_rent'
    test_rent['dataset'] = 'test_rent'
    
    # 统一列名
    train_rent.rename(columns=Config.col_mapping_rent, inplace=True)
    test_rent.rename(columns=Config.col_mapping_rent, inplace=True)
    
    return train_price, train_rent, test_price, test_rent

# --- 基础清洗工具函数 ---
def parse_floor_price(floor_str):
    if pd.isnull(floor_str): return np.nan, np.nan  
    pos_map = {'低': 1, '中': 2, '高': 3, '底': 0, '顶': 4}
    pos = np.nan
    for k, v in pos_map.items():
        if k in str(floor_str):
            pos = v
            break
    total_floor = re.findall(r"共(\d+)层", str(floor_str))
    total = int(total_floor[0]) if total_floor else np.nan
    return pos, total
    
def parse_floor_rent(floor_str):
    if pd.isnull(floor_str): return np.nan, np.nan, np.nan
    floor_level_map = {'低': 1, '中': 2, '高': 3, '底': 0, '顶': 4}
    level = np.nan
    for key, val in floor_level_map.items():
        if key in str(floor_str):
            level = val
            break
    total_floor_match = re.search(r'/(\d+)层', str(floor_str))
    if not total_floor_match:
        total_floor_match = re.search(r'共(\d+)层', str(floor_str))
    total_floor = float(total_floor_match.group(1)) if total_floor_match else np.nan
    current_floor_match = re.search(r'^(\d+)/', str(floor_str))
    current_floor = float(current_floor_match.group(1)) if current_floor_match else np.nan
    return level, total_floor, current_floor

def clean_numeric_text(text_str):
    if pd.isnull(text_str): return np.nan
    text_str = str(text_str)
    if '%' in text_str:
        val = re.search(r'([\d\.]+)', text_str)
        return float(val.group(1)) / 100.0 if val else np.nan
    if '-' in text_str:
        vals = re.findall(r'([\d\.]+)', text_str)
        if len(vals) >= 2: return (float(vals[0]) + float(vals[1])) / 2
        elif len(vals) == 1: return float(vals[0])
    val = re.search(r'([\d\.]+)', text_str)
    return float(val.group(1)) if val else np.nan

def clean_parking(parking_str):
    if pd.isnull(parking_str): return np.nan
    if '免费' in str(parking_str): return 0
    return clean_numeric_text(parking_str)

def parse_layout_price(layout_str):
    if pd.isnull(layout_str): return [np.nan] * 4
    room = re.search(r'(\d+)室', str(layout_str)); hall = re.search(r'(\d+)厅', str(layout_str))
    kitchen = re.search(r'(\d+)厨', str(layout_str)); bath = re.search(r'(\d+)卫', str(layout_str))
    return [float(room.group(1)) if room else 0, float(hall.group(1)) if hall else 0,
            float(kitchen.group(1)) if kitchen else 0, float(bath.group(1)) if bath else 0]

def parse_layout_rent(layout_str):
    if pd.isnull(layout_str): return [np.nan] * 3
    room = re.search(r'(\d+)室', str(layout_str)); hall = re.search(r'(\d+)厅', str(layout_str))
    bath = re.search(r'(\d+)卫', str(layout_str))
    return [float(room.group(1)) if room else 0, float(hall.group(1)) if hall else 0,
            float(bath.group(1)) if bath else 0]

def parse_area_price(area_str):
    if pd.isnull(area_str): return np.nan
    area_match = re.search(r'([\d\.]+)', str(area_str))
    return float(area_match.group(1)) if area_match else np.nan

def parse_ratio_price(ratio_str):
    if pd.isna(ratio_str) or ratio_str == '': return np.nan, np.nan
    ratio_str = str(ratio_str)
    chinese_map = {'一': '1', '二': '2', '两': '2', '三': '3', '四': '4', 
                   '五': '5', '六': '6', '七': '7', '八': '8', '九': '9', '十': '10'}
    for ch, num in chinese_map.items():
        ratio_str = ratio_str.replace(ch, num)
    numbers = re.findall(r'\d+', ratio_str)
    if len(numbers) >= 2: return float(numbers[0]), float(numbers[1])
    elif len(numbers) == 1: return float(numbers[0]), 1
    else: return np.nan, np.nan

def clean_year_price(year_str):
    if pd.isnull(year_str): return np.nan
    year_str = str(year_str).replace('年', '')
    if '-' in year_str:
        years = re.findall(r'(\d{4})', year_str)
        if len(years) >= 2: return (float(years[0]) + float(years[1])) / 2
    year_match = re.search(r'(\d{4})', year_str)
    return float(year_match.group(1)) if year_match else np.nan

def parse_area_rent(area_str):
    if pd.isnull(area_str): return np.nan
    area_match = re.search(r'([\d\.]+)', str(area_str))
    return float(area_match.group(1)) if area_match else np.nan

def parse_facilities_rent(facilities_str):
    if pd.isnull(facilities_str): return 0
    return len(str(facilities_str).split('、'))

def parse_ring_line(x):
    if pd.isna(x): return np.nan
    x = str(x)
    if '二环以内' in x: return 1
    if '二至三' in x: return 2
    if '三至四' in x: return 3
    if '四至五' in x: return 4
    if '五至六' in x: return 5
    if '六环以外' in x: return 6
    return np.nan


In [4]:

class SmartImputer:
    @staticmethod
    def hierarchical_fill(df, target_col, group_cols=['板块', '区域', '城市']):
        temp_df = df.copy()
        for g in group_cols:
            if g in temp_df.columns:
                temp_df[target_col] = temp_df[target_col].fillna(
                    temp_df.groupby(g)[target_col].transform('median')
                )
        temp_df[target_col] = temp_df[target_col].fillna(temp_df[target_col].median())
        return temp_df[target_col]

    @staticmethod
    def spatial_knn_fill(df, target_cols, n_neighbors=5):
        coords_cols = ['lon', 'lat']
        df['lon'] = df['lon'].fillna(df['lon'].median())
        df['lat'] = df['lat'].fillna(df['lat'].median())
        data_to_impute = df[coords_cols + target_cols].copy()
        imputer = KNNImputer(n_neighbors=n_neighbors)
        imputed_data = imputer.fit_transform(data_to_impute)
        return pd.DataFrame(imputed_data, columns=coords_cols + target_cols, index=df.index)[target_cols]


In [5]:
# ==========================================
# 核心数据处理器类
# ==========================================
class AdvancedDataProcessor:
    def __init__(self, train_df, test_df, task_type='price'):
        self.raw_train = train_df.copy()
        self.raw_test = test_df.copy()
        self.task_type = task_type
        self.df = pd.concat([self.raw_train, self.raw_test], axis=0).reset_index(drop=True)
        self.df['is_train'] = self.df['dataset'].apply(lambda x: 'train' in str(x))
        
    def preprocess_common_cols(self):
        print(f"[{self.task_type}] Processing Common Numeric Text Columns...")
        cols_to_clean = ['绿 化 率', '容 积 率', '物 业 费', '停车费用']
        for col in cols_to_clean:
            if col in self.df.columns:
                func = clean_parking if col == '停车费用' else clean_numeric_text
                self.df[col] = self.df[col].apply(func)
        if '停车位' in self.df.columns:
            self.df['ParkingSpots'] = self.df['停车位'].apply(clean_numeric_text)
        if '交易时间' in self.df.columns:
            self.df['TransactionTime'] = pd.to_datetime(self.df['交易时间'], errors='coerce')
            self.df['Trans_Year'] = self.df['TransactionTime'].dt.year
            self.df['Trans_Month'] = self.df['TransactionTime'].dt.month
            
        if '环线' in self.df.columns:
            print(f"[{self.task_type}] Parsing Ring Line info...")
            self.df['Ring_Line'] = self.df['环线'].apply(parse_ring_line)
            self.df['Ring_Line'] = self.df['Ring_Line'].fillna(self.df['Ring_Line'].mode()[0])
            
        return self

    def preprocess_specific(self):
        print(f"[{self.task_type}] Processing Task-Specific Columns...")
        if self.task_type == 'price':
            self.df[['Room', 'Hall', 'Kitchen', 'Bath']] = self.df['房屋户型'].apply(lambda x: pd.Series(parse_layout_price(x)))
            self.df[['Floor_Pos', 'Total_Floors']] = self.df['所在楼层'].apply(lambda x: pd.Series(parse_floor_price(x)))
            self.df['InnerArea'] = self.df['套内面积'].apply(parse_area_price)
            self.df['BuildingArea'] = self.df['建筑面积'].apply(parse_area_price)
            self.df['InnerAreaRatio'] = self.df['InnerArea'] / (self.df['BuildingArea'] + 1e-6)
            self.df[['Elevator_Num', 'House_Num']] = self.df['梯户比例'].apply(lambda x: pd.Series(parse_ratio_price(x)))
            self.df['Elevator_Ratio'] = self.df['Elevator_Num'] / (self.df['House_Num'] + 1e-6)
            self.df['Build_Year'] = self.df['建筑年代'].apply(clean_year_price)
            self.df['Area'] = self.df['BuildingArea'] # Alias
            
        elif self.task_type == 'rent':
            self.df[['Room', 'Hall', 'Bath']] = self.df['房屋户型'].apply(lambda x: pd.Series(parse_layout_rent(x)))
            self.df['Kitchen'] = 0
            self.df[['Floor_Pos', 'Total_Floors', 'Current_Floor']] = self.df['所在楼层'].apply(lambda x: pd.Series(parse_floor_rent(x)))
            self.df['Floor_Ratio'] = self.df['Floor_Pos'] / (self.df['Total_Floors'] + 1e-6)
            self.df['Area'] = self.df['建筑面积'].apply(parse_area_rent)
            self.df['Facilities_Count'] = self.df['配套设施'].apply(parse_facilities_rent)
            self.df['Build_Year'] = self.df['建筑年代'].apply(clean_year_price)
        return self

    def feature_engineering(self):
        print(f"[{self.task_type}] Engineering Features (with Ring & Interactions)...")
        
        self.df['lon'] = self.df['lon'].fillna(self.df['lon'].median())
        self.df['lat'] = self.df['lat'].fillna(self.df['lat'].median())

        physical_features = ['Build_Year', 'Total_Floors', '容 积 率', '绿 化 率']
        physical_features = [f for f in physical_features if f in self.df.columns]
        self.df[physical_features] = SmartImputer.spatial_knn_fill(self.df, physical_features)
        
        price_features = ['物 业 费', '停车费用']
        price_features = [f for f in price_features if f in self.df.columns]
        for col in price_features:
            self.df[col] = SmartImputer.hierarchical_fill(self.df, col, group_cols=['板块', '区域'])
    
        self.df['House_Age'] = self.df['Trans_Year'] - self.df['Build_Year']
        self.df['House_Age'] = self.df['House_Age'].apply(lambda x: max(0, x))
        
        kmeans = KMeans(n_clusters=30, random_state=42, n_init=10)
        self.df['Geo_Cluster'] = kmeans.fit_predict(self.df[['lon', 'lat']])
        self.df['Total_Rooms'] = self.df['Room'] + self.df['Hall'] + self.df['Bath']
        
        self.df['Avg_Room_Area'] = self.df['Area'] / (self.df['Total_Rooms'] + 1e-6)
        
        if 'Ring_Line' in self.df.columns:
            self.df['Ring_Area_Inter'] = self.df['Ring_Line'] * self.df['Area']
        
        self.extract_sentiment_features()
        return self

    def extract_sentiment_features(self):
        """基于字典的情感分析"""
        print(f"[{self.task_type}] Extracting Dictionary-based Sentiment...")
        neg_words = ['吵', '旧', '差', '堵', '乱', '窄', '偏', '硬', '薄', '老', '脏', '阴', '暗', '破', '远', '潮', '异味', '拥挤', '老化', '不足', '遗憾']
        pos_words = ['好', '亮', '静', '宽', '新', '优', '全', '顺', '美', '暖', '净', '通透', '便利', '齐全', '舒适', '方正', '合理', '充足', '到位']
        
        def _get_score(text):
            if pd.isna(text): return 0
            text = str(text)
            score = 0
            for w in neg_words:
                if w in text: score -= 1
            for w in pos_words:
                if w in text: score += 1
            return score

        target_cols = ['客户反馈', '核心卖点', '房屋优势'] if self.task_type == 'price' else ['客户反馈', '配套设施']
        for col in target_cols:
            if col in self.df.columns:
                self.df[f'{col}_senti_score'] = self.df[col].apply(_get_score)

    def nlp_features(self, text_cols, n_components=5):
        """NLP Features"""
        print(f"[{self.task_type}] Generating NLP Features for {text_cols}...")
        for col in text_cols:
            if col not in self.df.columns: continue
            
            self.df[f'{col}_len'] = self.df[col].astype(str).apply(len)
            
            tfidf = TfidfVectorizer(analyzer='char', ngram_range=(1, 4), max_features=5000, min_df=3)
            text_data = self.df[col].fillna("MISSING").astype(str)
            tfidf_matrix = tfidf.fit_transform(text_data)
            
            svd = TruncatedSVD(n_components=n_components, random_state=42)
            svd_vec = svd.fit_transform(tfidf_matrix)
            
            for i in range(n_components):
                self.df[f'{col}_tfidf_{i}'] = svd_vec[:, i]
        return self

    def finalize(self):
        drop_cols = ['房屋户型', '所在楼层', '建筑面积', '梯户比例', '建筑年代', '交易时间',
                     '户型', '楼层', '面积', '配套设施', 'TransactionTime', 'Price', 'dataset', '环线']
        
        final_df = self.df.copy()

        if 'Ring_Line' in final_df.columns:
            final_df['Ring_Line'] = final_df['Ring_Line'].astype(float)

        cat_cols = ['区域', '板块', '房屋朝向', '建筑结构', '装修情况', '物业类别', 'Geo_Cluster']
        cat_cols = [c for c in cat_cols if c in final_df.columns]
        
        le = LabelEncoder()
        for col in cat_cols:
            final_df[col] = final_df[col].astype(str)
            final_df[col] = le.fit_transform(final_df[col])
            
        numeric_cols = final_df.select_dtypes(include=[np.number]).columns.tolist()
        exclude = ['Price', 'ID', 'is_train']
        feature_cols = [c for c in numeric_cols if c not in exclude]
        
        train_out = final_df[final_df['is_train'] == True].copy()
        test_out = final_df[final_df['is_train'] == False].copy()
        
        y_train = train_out['Price']
        X_train = train_out[feature_cols]
        X_test = test_out[feature_cols]
        test_ids = test_out['ID']
        
        return X_train, y_train, X_test, test_ids, feature_cols


In [6]:
# ==========================================
# 神经网络模型
# ==========================================
class AdvancedMLP(nn.Module):
    def __init__(self, input_dim):
        super(AdvancedMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1), 
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.net(x)


In [7]:
# ==========================================
# 训练与模型融合类
# ==========================================
class RobustTrainer:
    def __init__(self, X, y, X_test, task_name='Price', test_ids=None):
        self.X = X.reset_index(drop=True)
        self.y = y.reset_index(drop=True)
        self.X_test = X_test.reset_index(drop=True)
        self.task_name = task_name
        self.test_ids = test_ids
        self.kf = KFold(n_splits=Config.n_folds, shuffle=True, random_state=Config.seed)
        self.oof_preds = {}
        self.test_preds = {}
        self.feature_importances = [] 

    def _log_metric(self, model_name, y_true_log, y_pred_log):
        y_true = np.expm1(y_true_log)
        y_pred = np.expm1(y_pred_log)
        y_pred = np.maximum(y_pred, 0)
        rmse = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
        mae = mean_absolute_error(y_true, y_pred)
        print(f" >> {model_name} | RMSE_Log: {rmse:.5f} | MAE_Orig: {mae:,.0f}")
        
    def _save_single_submission(self, model_name, preds_log):
        """保存单模型提交文件"""
        if self.test_ids is None: return
        preds = np.expm1(preds_log)
        preds = np.maximum(preds, 0)
        df = pd.DataFrame({'ID': self.test_ids, 'Price': preds})
        filename = f"submission_{self.task_name}_{model_name}.csv"
        df.to_csv(filename, index=False)
        print(f"    Saved: {filename}")

    def run_lgbm(self):
        print(f"Training LightGBM ({Config.LGBM_DEVICE})...")
        oof = np.zeros(len(self.X))
        test_pred = np.zeros(len(self.X_test))
        # 记录特征重要性
        imp_df = pd.DataFrame()
        imp_df['Feature'] = self.X.columns
        imp_df['Importance'] = 0
        
        params = {
            'objective': 'regression', 'metric': 'rmse', 
            'n_estimators': 8000, 
            'learning_rate': 0.01,
            'num_leaves': 42, 
            'colsample_bytree': 0.6,
            'device': Config.LGBM_DEVICE, 'random_state': Config.seed, 'verbose': -1
        }
        for fold, (tr_idx, val_idx) in enumerate(self.kf.split(self.X, self.y)):
            X_tr, y_tr = self.X.iloc[tr_idx], self.y.iloc[tr_idx]
            X_val, y_val = self.X.iloc[val_idx], self.y.iloc[val_idx]
            model = lgb.LGBMRegressor(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(200, verbose=False)])
            oof[val_idx] = model.predict(X_val)
            test_pred += model.predict(self.X_test) / Config.n_folds
            imp_df['Importance'] += model.feature_importances_ / Config.n_folds
            
        self.oof_preds['LGBM'] = oof
        self.test_preds['LGBM'] = test_pred
        self.feature_importances.append(imp_df) # 保存
        self._log_metric('LGBM', self.y, oof)
        self._save_single_submission('LGBM', test_pred)

    def run_xgb(self):
        print(f"Training XGBoost ({Config.XGB_DEVICE})...")
        oof = np.zeros(len(self.X))
        test_pred = np.zeros(len(self.X_test))
        params = {
            'objective': 'reg:squarederror', 'eval_metric': 'rmse',
            'learning_rate': 0.01, 
            'n_estimators': 8000,
            'max_depth': 8, 'subsample': 0.7, 'colsample_bytree': 0.6,
            'tree_method': Config.XGB_DEVICE, 'random_state': Config.seed
        }
        for fold, (tr_idx, val_idx) in enumerate(self.kf.split(self.X, self.y)):
            X_tr, y_tr = self.X.iloc[tr_idx], self.y.iloc[tr_idx]
            X_val, y_val = self.X.iloc[val_idx], self.y.iloc[val_idx]
            model = xgb.XGBRegressor(**params, early_stopping_rounds=200)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            oof[val_idx] = model.predict(X_val)
            test_pred += model.predict(self.X_test) / Config.n_folds
        self.oof_preds['XGB'] = oof
        self.test_preds['XGB'] = test_pred
        self._log_metric('XGB', self.y, oof)
        self._save_single_submission('XGB', test_pred)

    def run_nn(self):
        print(f"Training Neural Network ({Config.NN_DEVICE})...")
        # 使用 QuantileTransformer 处理长尾分布 将所有特征强制转换为高斯分布，对 NN 极其有效
        print("    Preprocessing: Applying QuantileTransformer...")
        scaler = QuantileTransformer(output_distribution='normal', random_state=42)
        
        X_fill = self.X.fillna(0)
        X_test_fill = self.X_test.fillna(0)
        X_scaled = scaler.fit_transform(X_fill)
        X_test_scaled = scaler.transform(X_test_fill)
        
        oof = np.zeros(len(self.X))
        test_pred = np.zeros(len(self.X_test))
        
        for fold, (tr_idx, val_idx) in enumerate(self.kf.split(X_scaled, self.y)):
            X_tr_t = torch.FloatTensor(X_scaled[tr_idx]).to(Config.NN_DEVICE)
            y_tr_t = torch.FloatTensor(self.y.iloc[tr_idx].values).view(-1, 1).to(Config.NN_DEVICE)
            X_val_t = torch.FloatTensor(X_scaled[val_idx]).to(Config.NN_DEVICE)
            y_val_t = torch.FloatTensor(self.y.iloc[val_idx].values).view(-1, 1).to(Config.NN_DEVICE)
            
            model = AdvancedMLP(X_scaled.shape[1]).to(Config.NN_DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=Config.NN_LR, weight_decay=1e-3)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=5)
            criterion = nn.HuberLoss() # 使用 HuberLoss，比 MSE 对异常值更不敏感
            
            best_loss = float('inf')
            early_stop_cnt = 0
            
            model.train()
            dataset = TensorDataset(X_tr_t, y_tr_t)
            loader = DataLoader(dataset, batch_size=Config.NN_BATCH_SIZE, shuffle=True)
            
            for epoch in range(Config.NN_EPOCHS):
                for bx, by in loader:
                    optimizer.zero_grad()
                    pred = model(bx)
                    loss = criterion(pred, by)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                
                model.eval()
                with torch.no_grad():
                    val_pred_t = model(X_val_t)
                    val_loss = criterion(val_pred_t, y_val_t).item()
                scheduler.step(val_loss)
                model.train()
                
                if val_loss < best_loss:
                    best_loss = val_loss
                    early_stop_cnt = 0
                    torch.save(model.state_dict(), f'nn_best_{self.task_name}_f{fold}.pth')
                else:
                    early_stop_cnt += 1
                    if early_stop_cnt >= 12: break
            
            model.load_state_dict(torch.load(f'nn_best_{self.task_name}_f{fold}.pth'))
            model.eval()
            with torch.no_grad():
                oof[val_idx] = model(X_val_t).cpu().numpy().flatten()
                test_tensor = torch.FloatTensor(X_test_scaled).to(Config.NN_DEVICE)
                test_pred += model(test_tensor).cpu().numpy().flatten() / Config.n_folds
        
        self.oof_preds['NN'] = oof
        self.test_preds['NN'] = test_pred
        self._log_metric('NN', self.y, oof)
        self._save_single_submission('NN', test_pred)

    def run_stacking(self):
        print("Running Stacking (Ridge Meta-Model)...")
        train_stack = pd.DataFrame(self.oof_preds)
        test_stack = pd.DataFrame(self.test_preds)
        
        meta = Ridge(alpha=1.0)
        meta.fit(train_stack, self.y)
        final_log = meta.predict(test_stack)
        
        stack_cv_pred = meta.predict(train_stack)
        self._log_metric('Stacking', self.y, stack_cv_pred)
        return np.expm1(final_log)

    def plot_feature_importance(self, top_n=20):
        if not self.feature_importances:
            print("No feature importance data found.")
            return
        
        # 聚合 Fold 结果
        df_imp = pd.concat(self.feature_importances).groupby('Feature')['Importance'].mean().reset_index()
        df_imp = df_imp.sort_values('Importance', ascending=False).head(top_n)
        
        plt.figure(figsize=(10, 8))
        sns.barplot(x='Importance', y='Feature', data=df_imp, palette='viridis')
        plt.title(f'LightGBM Feature Importance (Top {top_n}) - {self.task_name}')
        plt.tight_layout()
        plt.savefig(f'feature_importance_{self.task_name}.png')
        plt.show()


In [8]:
def main():
    train_p, train_r, test_p, test_r = load_data()
    
    # --- Price 任务 ---
    print("\n========== Price Data ==========")
    processor_p = AdvancedDataProcessor(train_p, test_p, task_type='price')
    processor_p.preprocess_common_cols().preprocess_specific().feature_engineering()
    processor_p.nlp_features(text_cols=['房屋优势','核心卖点','户型介绍','周边配套','交通出行','客户反馈'], n_components=5)
    
    X_train_p, y_train_p, X_test_p, ids_p, _ = processor_p.finalize()
    y_train_p = np.log1p(y_train_p)

    trainer_p = RobustTrainer(X_train_p, y_train_p, X_test_p, 'Price', test_ids=ids_p)
    trainer_p.run_lgbm()
    trainer_p.run_xgb()
    trainer_p.run_nn()
    pred_p = trainer_p.run_stacking()
    trainer_p.plot_feature_importance() 

    # --- Rent 任务 ---
    print("\n========== Rent Data ==========")
    processor_r = AdvancedDataProcessor(train_r, test_r, task_type='rent')
    processor_r.preprocess_common_cols().preprocess_specific().feature_engineering()
    processor_r.nlp_features(text_cols=['配套设施', '客户反馈'], n_components=5)
    
    X_train_r, y_train_r, X_test_r, ids_r, _ = processor_r.finalize()
    y_train_r = np.log1p(y_train_r)

    trainer_r = RobustTrainer(X_train_r, y_train_r, X_test_r, 'Rent', test_ids=ids_r)
    trainer_r.run_lgbm()
    trainer_r.run_xgb()
    trainer_r.run_nn()
    pred_r = trainer_r.run_stacking()
    trainer_r.plot_feature_importance()

    # --- 生成最终 Stacking 提交 ---
    sub_p = pd.DataFrame({'ID': ids_p, 'Price': pred_p})
    sub_r = pd.DataFrame({'ID': ids_r, 'Price': pred_r})
    final = pd.concat([sub_p, sub_r]).sort_values('ID')
    final['Price'] = final['Price'].clip(lower=0)
    final.to_csv('submission_stacking_final.csv', index=False)
    print("\nFinal Stacking Submission Saved: submission_stacking_final.csv")



In [9]:
if __name__ == "__main__":
    main()


========== Price Data ==========
[price] Processing Common Numeric Text Columns...
[price] Parsing Ring Line info...
[price] Processing Task-Specific Columns...
[price] Engineering Features (with Ring & Interactions)...
[price] Extracting Dictionary-based Sentiment...
[price] Generating NLP Features for ['房屋优势', '核心卖点', '户型介绍', '周边配套', '交通出行', '客户反馈']...
Training LightGBM (cpu)...
 >> LGBM | RMSE_Log: 0.12985 | MAE_Orig: 221,071
    Saved: submission_Price_LGBM.csv
Training XGBoost (gpu_hist)...
 >> XGB | RMSE_Log: 0.12263 | MAE_Orig: 206,321
    Saved: submission_Price_XGB.csv
Training Neural Network (cuda)...
    Preprocessing: Applying QuantileTransformer...
 >> NN | RMSE_Log: 0.28294 | MAE_Orig: 509,657
    Saved: submission_Price_NN.csv
Running Stacking (Ridge Meta-Model)...
 >> Stacking | RMSE_Log: 0.12169 | MAE_Orig: 204,628


<Figure size 1000x800 with 1 Axes>


========== Rent Data ==========
[rent] Processing Common Numeric Text Columns...
[rent] Parsing Ring Line info...
[rent] Processing Task-Specific Columns...
[rent] Engineering Features (with Ring & Interactions)...
[rent] Extracting Dictionary-based Sentiment...
[rent] Generating NLP Features for ['配套设施', '客户反馈']...
Training LightGBM (cpu)...
 >> LGBM | RMSE_Log: 0.17040 | MAE_Orig: 68,296
    Saved: submission_Rent_LGBM.csv
Training XGBoost (gpu_hist)...
 >> XGB | RMSE_Log: 0.16617 | MAE_Orig: 65,876
    Saved: submission_Rent_XGB.csv
Training Neural Network (cuda)...
    Preprocessing: Applying QuantileTransformer...
 >> NN | RMSE_Log: 0.29208 | MAE_Orig: 130,814
    Saved: submission_Rent_NN.csv
Running Stacking (Ridge Meta-Model)...
 >> Stacking | RMSE_Log: 0.16576 | MAE_Orig: 65,664


<Figure size 1000x800 with 1 Axes>


Final Stacking Submission Saved: submission_stacking_final.csv


In [10]:
MODEL_LIST = ['LGBM', 'XGB', 'NN']
OUTPUT_PREFIX = 'submission4_'

for model_name in MODEL_LIST:
    price_file = f"submission_Price_{model_name}.csv"
    rent_file = f"submission_Rent_{model_name}.csv"
    
    if not os.path.exists(price_file):
        print(f"缺失文件：{price_file}，跳过该模型")
        continue
    if not os.path.exists(rent_file):
        print(f"缺失文件：{rent_file}，跳过该模型")
        continue
    
    df_price = pd.read_csv(price_file)
    df_rent = pd.read_csv(rent_file)
    df_combined = pd.concat([df_price, df_rent], axis=0)
    df_combined = df_combined.sort_values('ID')
    df_combined['Price'] = df_combined['Price'].clip(lower=0) 
    
    output_file = f"{OUTPUT_PREFIX}{model_name}.csv"
    df_combined.to_csv(output_file, index=False)
    
